# 04A — Audit de référence des modèles SIMCA sélectionnés

Ce notebook est la suite directe de 03B–03C. Il ne constitue pas une seconde recherche de modèles : **03B reste l’unique autorité de sélection**. 04A extrait en flux les seules politiques de seuil retenues, reconstruit leurs métriques de validation croisée avec la logique exacte de 03B et vérifie leur concordance avec `model_metrics.parquet`.

Aucun modèle n’est réajusté, aucun seuil n’est proposé et aucun nouvel identifiant n’est créé. Le statut 03C détermine seulement l’usage aval (`supported` ou `diagnostic_only`) ; les tracks non soutenus restent visibles dans l’audit.

## A — Initialisation et chemins centralisés

In [1]:
from __future__ import annotations

import json
import sys
from pathlib import Path

import pandas as pd
from IPython.display import display

current_dir = Path.cwd().resolve()
if (current_dir / "src").is_dir():
    PROJECT_ROOT = current_dir
elif (current_dir.parent / "src").is_dir():
    PROJECT_ROOT = current_dir.parent
else:
    raise RuntimeError("Launch 04A from the repository or notebooks directory.")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src import experiment_config as expcfg
from src.protocol_governance import (
    sha256_file,
    sha256_payload,
    verify_frozen_protocol,
    resolve_protocol_for_execution,
)
from src.utils import load_parquet, save_parquet
from src.workflows.protocol_audit import assert_no_forbidden_score_columns
from src.workflows.simca_calibration_registry import (
    validate_internal_calibration_manifest,
)
from src.workflows.simca_selected_model_audit import (
    run_selected_model_reference_audit,
)

In [2]:
results_tag = (
    f"{int(expcfg.WAVELENGTH_WINDOW_MIN_NM)}_{int(expcfg.WAVELENGTH_WINDOW_MAX_NM)}"
    if expcfg.USE_WAVELENGTH_WINDOW
    else expcfg.DEFAULT_RESULTS_TAG
)
protocol_dir = PROJECT_ROOT.joinpath(*expcfg.PROTOCOL_ARTIFACT_RELATIVE_DIR)
input_dir_03b = (
    PROJECT_ROOT
    / "results"
    / f"{expcfg.INTERNAL_CALIBRATION_RESULTS_DIR_PREFIX}_{results_tag}"
)
input_dir_03c = (
    PROJECT_ROOT
    / "results"
    / f"{expcfg.DOMAIN_SPATIAL_CALIBRATION_RESULTS_DIR_PREFIX}_{results_tag}"
)
output_dir = (
    PROJECT_ROOT
    / "results"
    / f"{expcfg.SIMCA_GRID_SEARCH_RESULTS_DIR_PREFIX}_{results_tag}"
)
output_dir.mkdir(parents=True, exist_ok=True)

input_paths_03b = {
    key: input_dir_03b / expcfg.INTERNAL_CALIBRATION_OUTPUT_FILENAMES[key]
    for key in expcfg.SIMCA_GRID_REQUIRED_03B_ARTIFACTS
}
input_manifest_path_03b = (
    input_dir_03b
    / expcfg.INTERNAL_CALIBRATION_OUTPUT_FILENAMES["checkpoint_manifest"]
)
input_paths_03c = {
    key: input_dir_03c / expcfg.DOMAIN_SPATIAL_CALIBRATION_OUTPUT_FILENAMES[key]
    for key in expcfg.SIMCA_GRID_REQUIRED_03C_ARTIFACTS
}
output_paths = {
    key: output_dir / filename
    for key, filename in expcfg.SIMCA_GRID_SEARCH_OUTPUT_FILENAMES.items()
}

print("03B:", input_dir_03b)
print("03C:", input_dir_03c)
print("04A:", output_dir)

03B: C:\Users\alixg\OneDrive - Université Paris-Dauphine\hsi_nuts\results\03B_internal_calibration_8tracks_v5_px_qc_v1
03C: C:\Users\alixg\OneDrive - Université Paris-Dauphine\hsi_nuts\results\03C_projection_spatial_calibration_8tracks_v5_px_qc_v1
04A: C:\Users\alixg\OneDrive - Université Paris-Dauphine\hsi_nuts\results\04A_simca_grid_search_8tracks_v5_px_qc_v1


## B — Vérification bloquante des contrats 03B–03C

In [3]:
# verify_frozen_protocol(protocol_dir, strict=True)
# protocol_lock = json.loads(
#     (protocol_dir / expcfg.PROTOCOL_OUTPUT_FILENAMES["lock"]).read_text(
#         encoding="utf-8"
#     )
# )
# protocol_hash = str(protocol_lock["lock_sha256"])
protocol_checks_df, protocol_hash = (
    resolve_protocol_for_execution(
        protocol_dir,
        upstream_manifest_path=input_manifest_path_03b,
    )
)
if not input_manifest_path_03b.is_file():
    raise FileNotFoundError(input_manifest_path_03b)
manifest_03b = json.loads(input_manifest_path_03b.read_text(encoding="utf-8"))
validate_internal_calibration_manifest(
    manifest_03b,
    input_paths_03b,
    required_artifacts=expcfg.SIMCA_GRID_REQUIRED_03B_ARTIFACTS,
    protocol_hash=protocol_hash,
)
artifact_hashes_03b = {
    str(entry["name"]): str(entry["sha256"])
    for entry in manifest_03b["artifacts"]
}

manifest_03c = json.loads(
    input_paths_03c["audit_manifest"].read_text(encoding="utf-8")
)
if str(manifest_03c["protocol_hash"]) != protocol_hash:
    raise RuntimeError("03C and the frozen protocol have different hashes.")
if manifest_03c["input_03b_manifest_sha256"] != sha256_file(input_manifest_path_03b):
    raise RuntimeError("03C was not built from the current 03B manifest.")
eligibility_entry = manifest_03c["output_artifacts"]["projection_eligibility"]
if eligibility_entry["sha256"] != sha256_file(input_paths_03c["projection_eligibility"]):
    raise RuntimeError("03C projection eligibility hash mismatch.")
if manifest_03c["spatial_postprocessing_lock_sha256"] != sha256_file(
    input_paths_03c["spatial_postprocessing_lock"]
):
    raise RuntimeError("03C spatial lock hash mismatch.")
spatial_lock = json.loads(
    input_paths_03c["spatial_postprocessing_lock"].read_text(encoding="utf-8")
)
if str(spatial_lock["protocol_hash"]) != protocol_hash:
    raise RuntimeError("The spatial lock does not match the frozen protocol.")

track_contracts = load_parquet(input_paths_03b["track_contracts"])
model_catalog = load_parquet(input_paths_03b["model_catalog"])
selected_models = load_parquet(input_paths_03b["selected_models"])
selected_runs = load_parquet(input_paths_03b["selected_runs"])
selected_threshold_rows = load_parquet(input_paths_03b["selected_thresholds"])
model_metrics = load_parquet(input_paths_03b["model_metrics"])
projection_eligibility = load_parquet(input_paths_03c["projection_eligibility"])
expected_track_ids = set(track_contracts["track_id"].astype(str))
if expected_track_ids != {f"E{index}" for index in range(1, 9)}:
    raise RuntimeError("The 03B contract must contain exactly tracks E1-E8.")
if set(projection_eligibility["track_id"].astype(str)) != expected_track_ids:
    raise RuntimeError("03C eligibility does not cover the eight 03B tracks.")

if manifest_03c.get("spatial_selection_scope") != expcfg.SPATIAL_CALIBRATION_SELECTION_SCOPE:
    raise RuntimeError("03C manifest does not declare within-track spatial selection.")
if manifest_03c.get("spatial_selection_policy") != expcfg.SPATIAL_CALIBRATION_SELECTION_POLICY:
    raise RuntimeError("03C manifest has an unexpected spatial selection policy.")
if spatial_lock.get("selection_scope") != expcfg.SPATIAL_CALIBRATION_SELECTION_SCOPE:
    raise RuntimeError("03C spatial lock is not within-track.")
if spatial_lock.get("selection_policy") != expcfg.SPATIAL_CALIBRATION_SELECTION_POLICY:
    raise RuntimeError("03C spatial lock policy is inconsistent with experiment_config.")
if "selected_parameters" in spatial_lock or "selection_weighting" in spatial_lock:
    raise RuntimeError("Legacy global spatial-lock fields are forbidden.")

expected_spatial_tracks = set(
    projection_eligibility.loc[
        projection_eligibility["track_id"].astype(str).isin(
            expcfg.SPATIAL_CALIBRATION_PIXEL_TRACK_IDS
        )
        & projection_eligibility["eligibility_status"].astype(str).isin(
            expcfg.PROJECTION_DOMAIN_SPATIAL_SUPPORTED_STATUSES
        ),
        "track_id",
    ].astype(str)
)
locked_spatial_tracks = set(
    map(str, spatial_lock.get("selected_parameters_by_track", {}))
)
if locked_spatial_tracks != expected_spatial_tracks:
    raise RuntimeError(
        "03C spatial-lock tracks do not match supported pixel tracks: "
        f"expected={sorted(expected_spatial_tracks)}, "
        f"locked={sorted(locked_spatial_tracks)}."
    )

display(
    projection_eligibility[["track_id", "eligibility_status", "eligibility_reason"]]
)

,track_id,eligibility_status,eligibility_reason
0,E1,eligible,all_predeclared_limits_satisfied
1,E2,eligible,all_predeclared_limits_satisfied
2,E3,unsupported_domain_shift,standardized_shift
3,E4,unsupported_domain_shift,standardized_shift
4,E5,eligible,all_predeclared_limits_satisfied
5,E6,eligible_with_warning,out_of_domain_rate;target_rejection_rate
6,E7,eligible,all_predeclared_limits_satisfied
7,E8,eligible_with_warning,out_of_domain_rate;target_rejection_rate


## C — Audit vectorisé des politiques sélectionnées

Le fichier volumineux `threshold_metrics.parquet` est filtré par le moteur Arrow avant conversion en pandas. Seules les coordonnées naturelles `(model_id, random_state, decision_scope, quantiles/vote)` retenues en 03B sont agrégées. La comparaison finale réutilise `aggregate_threshold_candidates` et `build_model_metrics`, les fonctions qui ont produit la sélection 03B.

In [4]:
if expcfg.SIMCA_GRID_SEARCH_RUN:
    audit_outputs = run_selected_model_reference_audit(
        model_catalog=model_catalog,
        selected_models=selected_models,
        selected_runs=selected_runs,
        selected_threshold_rows=selected_threshold_rows,
        model_metrics=model_metrics,
        track_contracts=track_contracts,
        projection_eligibility=projection_eligibility,
        threshold_metrics_path=input_paths_03b["threshold_metrics"],
    )
else:
    audit_outputs = {
        key: load_parquet(output_paths[key])
        for key in ("model_reference", "fold_metrics")
    }

model_reference = audit_outputs["model_reference"]
fold_metrics = audit_outputs["fold_metrics"]
if tuple(model_reference.columns) != expcfg.SIMCA_GRID_MODEL_REFERENCE_COLUMNS:
    raise RuntimeError("Unexpected 04A model-reference schema.")
if tuple(fold_metrics.columns) != expcfg.SIMCA_GRID_SELECTED_FOLD_METRIC_COLUMNS:
    raise RuntimeError("Unexpected 04A fold-metric schema.")
if model_reference["model_id"].duplicated().any():
    raise RuntimeError("04A must contain one model-reference row per model_id.")
fold_key = ["model_id", "random_state", "decision_scope", "fold_id"]
if fold_metrics.duplicated(fold_key).any():
    raise RuntimeError("04A fold natural keys are not unique.")
if set(model_reference["model_id"].astype(str)) != set(
    selected_models["model_id"].astype(str)
):
    raise RuntimeError("04A changed the 03B selected-model universe.")

display(
    model_reference.groupby(
        ["track_id", "eligibility_status", "downstream_status"],
        as_index=False,
    ).agg(
        n_models=("model_id", "size"),
        n_runs=("n_selected_runs", "sum"),
        max_metric_difference=("max_abs_metric_difference", "max"),
    )
)
display(assert_no_forbidden_score_columns(audit_outputs))

,track_id,eligibility_status,downstream_status,n_models,n_runs,max_metric_difference
0,E1,eligible,supported,2,2,2.850657e-08
1,E2,eligible,supported,8,8,2.750984e-08
2,E3,unsupported_domain_shift,diagnostic_only,1,1,1.833989e-08
3,E4,unsupported_domain_shift,diagnostic_only,1,1,1.083721e-08
4,E5,eligible,supported,1,3,2.188871e-08
5,E6,eligible_with_warning,supported,17,39,2.921283e-08
6,E7,eligible,supported,1,3,2.188871e-08
7,E8,eligible_with_warning,supported,7,21,2.817939e-08


,table,n_columns,forbidden_score_columns,score_free
0,model_reference,7,,True
1,fold_metrics,17,,True


## D — Persistance compacte et provenance

In [5]:
if expcfg.SIMCA_GRID_SEARCH_RUN:
    save_parquet(model_reference, output_paths["model_reference"], optimize=False)
    save_parquet(fold_metrics, output_paths["fold_metrics"], optimize=False)

output_artifacts = {
    key: {
        "path": str(output_paths[key]),
        "row_count": int(len(table)),
        "columns": list(table.columns),
        "sha256": sha256_file(output_paths[key]),
    }
    for key, table in audit_outputs.items()
}
audit_manifest = {
    "notebook": "04A_simca_grid_search",
    "audit_contract": "selected_model_reference_audit_v1",
    "protocol_version": str(expcfg.PROTOCOL_VERSION),
    "schema_version": str(expcfg.RESULTS_SCHEMA_VERSION),
    "protocol_hash": protocol_hash,
    "selection_authority": "03B_selected_models",
    "selection_mutated": False,
    "model_refit": False,
    "threshold_resuggestion": False,
    "weighted_score_used": False,
    "natural_execution_key": ["model_id", "random_state"],
    "spatial_selection_scope": str(spatial_lock["selection_scope"]),
    "spatial_selection_policy": str(spatial_lock["selection_policy"]),
    "spatial_track_ids": list(map(str, spatial_lock["spatial_track_ids"])),
    "natural_fold_key": [
        "model_id", "random_state", "decision_scope", "fold_id"
    ],
    "unsupported_track_policy": "retained_as_diagnostic_only",
    "reference_metric_atol": float(expcfg.SIMCA_GRID_REFERENCE_METRIC_ATOL),
    "max_abs_metric_difference": float(
        model_reference["max_abs_metric_difference"].max()
    ),
    "input_03b_manifest_sha256": sha256_file(input_manifest_path_03b),
    "input_03c_manifest_sha256": sha256_file(
        input_paths_03c["audit_manifest"]
    ),
    "input_sha256": {
        **{
            f"03B.{key}": artifact_hashes_03b[key]
            for key in expcfg.SIMCA_GRID_REQUIRED_03B_ARTIFACTS
        },
        "03C.projection_eligibility": str(eligibility_entry["sha256"]),
        "03C.spatial_postprocessing_lock": str(
            manifest_03c["spatial_postprocessing_lock_sha256"]
        ),
        "03C.audit_manifest": sha256_file(input_paths_03c["audit_manifest"]),
    },
    "output_artifacts": output_artifacts,
    "counts": {
        "selected_models": int(len(model_reference)),
        "selected_runs": int(
            selected_runs[["model_id", "random_state"]].drop_duplicates().shape[0]
        ),
        "selected_run_scopes": int(
            fold_metrics[["model_id", "random_state", "decision_scope"]]
            .drop_duplicates()
            .shape[0]
        ),
        "run_fold_rows": int(len(fold_metrics)),
        "supported_models": int(model_reference["downstream_status"].eq("supported").sum()),
        "diagnostic_only_models": int(
            model_reference["downstream_status"].eq("diagnostic_only").sum()
        ),
    },
}
audit_manifest["manifest_payload_sha256"] = sha256_payload(audit_manifest)
output_paths["audit_manifest"].write_text(
    json.dumps(audit_manifest, indent=2, ensure_ascii=False),
    encoding="utf-8",
)

print("Saved outputs:")
for path in output_paths.values():
    print(" -", path)

Saved outputs:
 - C:\Users\alixg\OneDrive - Université Paris-Dauphine\hsi_nuts\results\04A_simca_grid_search_8tracks_v5_px_qc_v1\selected_model_reference.parquet
 - C:\Users\alixg\OneDrive - Université Paris-Dauphine\hsi_nuts\results\04A_simca_grid_search_8tracks_v5_px_qc_v1\selected_run_fold_metrics.parquet
 - C:\Users\alixg\OneDrive - Université Paris-Dauphine\hsi_nuts\results\04A_simca_grid_search_8tracks_v5_px_qc_v1\audit_manifest.json


## Lecture des sorties

- `selected_model_reference.parquet` : une ligne par `model_id`, avec le track, les cardinalités naturelles, le statut de domaine 03C et l’écart maximal de reproduction des métriques 03B.
- `selected_run_fold_metrics.parquet` : une ligne par `(model_id, random_state, decision_scope, fold_id)` ; les seuils restent dans l’artefact 03B faisant autorité.
- `audit_manifest.json` : hashes des entrées/sorties et déclaration explicite que 04A n’a ni réajusté ni resélectionné les modèles.